# Clase 36 - Notebook 1 - Simulación del modelo de balances de masa y energía

El modelo del secador por aspersión consiste de un sistema de ecuaciones diferenciales más ecuaciones algebraicas (DAE).


### ⚙️ Paso 0: Configuración Automática del Entorno (Google Colab)
Si estás ejecutando este cuaderno en **Google Colab**, ejecuta la siguiente celda una sola vez al inicio de la sesión para clonar automáticamente el repositorio e instalar todas las dependencias requeridas.


In [ ]:
# --- Preámbulo Universal para Google Colab y Entornos Locales ---
import os, sys

if 'google.colab' in sys.modules:
    REPO_DIR = '/content/DAII-SprayDrying'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/felipehuerta17/DAII-SprayDrying.git {REPO_DIR}
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    %pip install -q numpy scipy pandas matplotlib pymoo casadi
    print('✅ Entorno de Google Colab configurado con éxito.')
else:
    print('✅ Ejecutando en entorno local.')


# Modelo de Ecuaciones Diferenciales Algebraicas (DAE) del Secador Spray

Este documento describe el modelo matemático implementado en `spraydrylib`, incluyendo todas las constantes, propiedades físicas, ecuaciones algebraicas y ecuaciones diferenciales.

---

## Variables de estado
- $H_o(t)$: Humedad de salida del aire (kg vapor / kg aire seco)
- $X_o(t)$: Contenido de humedad de la partícula (kg agua / kg sólidos secos)
- $T_4(t)$: Temperatura del aire de salida (K)
- $r_d(t)$: Radio de la gota (m) [variable algebraica]

## Ecuaciones diferenciales (DAE)
1. **Evolución de humedad de salida del aire**:
$$\frac{dH_o}{dt} = \frac{G}{M_A}(H_i - H_o) + \frac{F}{M_A}(X_i - X_o)$$

2. **Evolución de contenido de humedad en el sólido**:
$$\frac{dX_o}{dt} = \begin{cases} \dfrac{h}{\lambda_{sat}} \dfrac{a_{drop}}{m_s} (T_{sat} - T_4), & X_o \geq X_{oc} \\[1.0em] -\dfrac{4\pi^2}{(2r_d)^2} D_e (X_o - m_{equil}H_o), & X_o < X_{oc} \end{cases}$$

3. **Evolución de la temperatura del aire de salida**:
$$\frac{dT_4}{dt} = \frac{F_1 c_{p,in}(T_{sat})(T_F - T_{sat}) + F_3 c_p(T_i, H_i)(T_i - T_{sat}) - F_4 c_p(T_4,H_o)(T_4 - T_{sat}) + \lambda_{sat}(T_{sat})(F_3 y_{v,3} - F_4 y_{v,4}) - \dot{Q}_{loss}}{M_g c_v(H_o)}$$


# 01 — Simulación Básica y Perfiles Temporales


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from spraydrylib.backends import get_simulator
from spraydrylib import model as mdl
import os; os.makedirs("./outputs", exist_ok=True)

simulate, backend = get_simulator()
print("Backend cargado:", backend)

p = mdl.ParamsExact(G=650/3600.0, T_i=145+273.15, Hi=0.0152,
                    F=4.5/3600, T_F=28+273.15, Xi=3.0, ri=5.0e-5)
sol = simulate(p, tf=400.0, n_steps=10000)
t, H, T4, Xo, rd = sol["t"], sol["Ho"], sol["T4"], sol["Xo"], sol["rd"]

# Humedad y Temperatura del aire
fig, ax1 = plt.subplots(figsize=(7,4))
ax1.plot(t, H, color='skyblue', label='Humedad', linewidth=2)
ax1.set_xlabel('Tiempo (s)', fontsize=12)
ax1.set_ylabel(r'Humedad del aire $\left(\frac{kg \,agua}{kg \,aire\,seco}\right)$', fontsize=12)
ax1.set_xlim([0, 400])
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.3f}"))
ax2 = ax1.twinx()
ax2.plot(t, T4, color='darkred', label='Temperatura', linewidth=2)
ax2.set_ylabel(r'Temperatura $(K)$', fontsize=12)
plt.title('Evolución de temperatura y humedad del aire', fontsize=14)
lines = ax1.get_lines() + ax2.get_lines(); labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right')
plt.tight_layout(); plt.savefig('./outputs/evolucion_temperatura_humedad.svg', format='svg'); plt.show()

# Radio de gota (tiempo corto)
mask = t <= 5
plt.figure(figsize=(6.4,3.8))
plt.plot(t[mask], rd[mask], linewidth=2)
plt.xlabel('Tiempo (s)'); plt.ylabel('Radio de gota (m)')
plt.title('Evolución del radio de la gota (tiempo corto)')
plt.tight_layout(); plt.savefig('./outputs/radio_gota_t_corto.svg', format='svg'); plt.show()


In [ ]:
# Figura: evolución de contenido de agua
plt.figure(figsize=(6.4,3.8))
plt.plot(t, Xo, color='royalblue', linewidth=2)
plt.axhline(min(Xo), color='k', linestyle='-.', linewidth=1.2)
plt.axhline(Xo[-1], color='k', linestyle='-.', linewidth=1.2)
plt.xlabel('Tiempo (s)', fontsize=12)
plt.ylabel(r'Contenido de agua $\left(\frac{kg \, agua}{kg \, sólidos}\right)$', fontsize=12)
plt.title('Evolución de contenido de agua:\nénfasis en etapa de tasa de secado decreciente', fontsize=13)
plt.ylim([0, 0.08])
plt.xlim([0, 200])
plt.tight_layout()
plt.savefig('./outputs/evolucion_contenido_agua.svg', format='svg', dpi=300)
plt.show()


---
## 🔬 Exploración y Modificación del Modelo Fenomenológico (API v2.0)
Con la nueva estructura modular de `spraydrylib`, los estudiantes pueden modificar fácilmente los parámetros del modelo, agregar fenómenos como **pérdidas de calor en pared** y **radiación térmica**, o cambiar las correlaciones de transporte sin modificar el código interno.


In [ ]:
# 1. Configuración orientada a objetos con SprayDryer
from spraydrylib import SprayDryer, OperatingConditions, DryerParameters

# Condiciones operativas personalizadas
op = OperatingConditions(
    G=650 / 3600.0,    # Caudal de aire seco (kg/s)
    T_i=145 + 273.15,  # Temperatura de entrada del aire (K)
    ri=5.0e-5          # Radio inicial de gota (50 um)
)

# Parámetros del secador con pérdidas por pared y radiación
params_dryer = DryerParameters(
    A_wall=15.0,        # Área de pared (m2)
    U_wall=5.0,         # Coeficiente global de pérdida en pared (W/m2/K)
    emissivity=0.85,    # Emisividad de radiación térmica
    Nu=2.0              # Número de Nusselt para la gota
)

dryer = SprayDryer(params=op, config=params_dryer)
res = dryer.simulate(tf=400.0, n_steps=600)

print(f"Temperatura final T4: {res.T4[-1] - 273.15:.2f} °C")
print(f"Humedad final del sólido Xo: {res.Xo[-1]:.4f} kg agua / kg sólido")
print(f"Gasto energético: {res.get_energy_consumption_kw():.2f} kW")

res.plot()


In [ ]:
# Comparación: Secador Adiabático vs Con Pérdidas de Calor y Radiación
dryer_adiab = SprayDryer(params=op, config=DryerParameters(U_wall=0.0, emissivity=0.0))
res_adiab = dryer_adiab.simulate(tf=400.0)

dryer_conv = SprayDryer(params=op, config=DryerParameters(U_wall=10.0, emissivity=0.0))
res_conv = dryer_conv.simulate(tf=400.0)

dryer_rad = SprayDryer(params=op, config=DryerParameters(U_wall=10.0, emissivity=0.85))
res_rad = dryer_rad.simulate(tf=400.0)

plt.figure(figsize=(7.5, 4.2))
plt.plot(res_adiab.t, res_adiab.T4 - 273.15, label="Adiabático", lw=2)
plt.plot(res_conv.t, res_conv.T4 - 273.15, label="Pérdida Pared (U=10)", lw=2)
plt.plot(res_rad.t, res_rad.T4 - 273.15, label="Pared + Radiación (eps=0.85)", lw=2)
plt.xlabel("Tiempo (s)")
plt.ylabel("Temperatura del aire salida T4 (°C)")
plt.title("Impacto de las Pérdidas Térmicas en la Cámara de Secado")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
